<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/8_parallel_simulation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8. 並列シミュレーション / Parallel Simulation on GPU

**目的 / Objective:** `scene.build(n_envs=N)` でN個の環境を並列に構築し、`envs_idx` というテンソルを使って全環境を一度のPython呼び出しで操作する方法を学びます。

Learn to build N parallel environments with `scene.build(n_envs=N)` and control all of them in a single Python call using an `envs_idx` tensor.

## Setup / セットアップ

以下のセルを実行して、依存パッケージをインストールし、GPUレンダリングを設定します。

Run the cell below to install dependencies and configure GPU rendering.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import urllib.request
    exec(urllib.request.urlopen(
        "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
    ).read(), globals())
    setup_colab()

import numpy as np
import torch
import genesis as gs

gs.init(backend=gs.gpu)
device = gs.device
print(f"Genesis device: {device}")

from hsr_genesis.hsr_rigid_entity import HSRBURDF
from hsr_genesis import tutorial_utils

## 1. Build N parallel environments / N個の並列環境を構築

単一環境との違いは `scene.build(n_envs=N)` を呼ぶことだけです。Genesisが内部でシーンの構成をN個分複製し、`build()` を実行した後は1つのPythonオブジェクトからN個の環境すべてを制御できます。

`scene.build(n_envs=N)` is the only difference from single-env. Genesis clones the entity graph N times internally. After `build()`, one Python handle controls all N envs.

In [ ]:
N = 8  # try 16, 32, 64 on Colab Pro

scene = gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.02),
    rigid_options=gs.options.RigidOptions(use_gjk_collision=True),
    show_viewer=False,
)
_ = scene.add_entity(gs.morphs.Plane())
hsr = scene.add_entity(
    HSRBURDF(file=str(tutorial_utils._find_urdf()), robot="hsrb",
             base_mode="planar",
             links_to_keep=["hand_palm_link"],
             end_effector_frame="hand_palm_link")
)
scene.build(n_envs=N, env_spacing=(3.0, 3.0))

# The mental switch: envs_idx is a tensor of env indices on gs.device.
envs_all = torch.arange(N, device=device, dtype=gs.tc_int)
print(f"Built {N} envs. envs_all = {envs_all}")

## 2. Tensors, batched tensors, and GPU memory / テンソル・バッチテンソル・GPUメモリ

### テンソルとは / What is a tensor?

**テンソル** とは多次元配列のことです。PyTorchの `torch.Tensor` はGPU上に置ける多次元配列で、GPU上で高速に並列計算されます。

A **tensor** is a multi-dimensional array. PyTorch's `torch.Tensor` is a multi-dimensional array that can be placed on GPU and computed in parallel by GPU kernels.

| 次元 / Dimension | 名前 / Name | 例 / Example | 形状 / Shape |
| --- | --- | --- | --- |
| 0次元 | スカラー / scalar | `5.0` | `()` |
| 1次元 | ベクトル / vector | `[1.0, 2.0, 3.0]` | `(3,)` |
| 2次元 | 行列 / matrix | 関節角度の集合 | `(N, dof)` |
| 3次元以上 | テンソル / tensor | 画像データなど | `(N, C, H, W)` |

チュートリアル1〜7では、ロボットの関節角度 `qpos` は `(dof,)` という1次元テンソルでした。これは1台のロボットが持つすべての関節値をまとめたリストです。

In tutorials 1–7, the robot's joint angles `qpos` was a 1-D tensor of shape `(dof,)` — a list of all joint values for one robot.

### バッチテンソルとは / What is a batched tensor?

**バッチテンソル** とは、複数のデータの先頭に新しい次元（バッチ次元）を追加してひとつのテンソルにまとめたものです。N台のロボットの `qpos` をまとめると、形状は `(dof,)` から `(N, dof)` に変わります。GPUはこのバッチ次元に沿って全データを並列に計算します。

A **batched tensor** stacks multiple data items along a new leading dimension (the batch dimension). Stacking N robots' `qpos` gives shape `(dof,)` → `(N, dof)`. The GPU computes all N items in parallel along this batch dimension.

```
Single-env qpos:          Batched qpos (N=8):
(8,)                      (8, 8)  ← batch dimension added

[0.1, 0.2, ...]          [[0.1, 0.2, ...]     ← env 0
                           [0.3, 0.1, ...]     ← env 1
                           [0.2, 0.2, ...]     ← env 2
                              ...
                           [0.1, 0.3, ...]]    ← env 7
```

`hsr.set_qpos(qpos, envs_idx=envs_all)` を1回呼ぶだけで、N個すべての環境の関節を同時に設定できます。

One `hsr.set_qpos(qpos, envs_idx=envs_all)` call sets all N envs' joints simultaneously.

### CPUとGPUはそれぞれ独自のメモリを持つ / CPU and GPU each have their own memory

CPU（ホスト）とGPU（デバイス）はそれぞれ独立したメモリを持っています。Python変数やNumPy配列はCPU側のメモリに置かれ、PyTorchのテンソルは `gs.device`（GPU側）に置くことができます。両者の間でデータをコピーするにはPCIeバスを経由する必要があり、これは計算そのものよりもはるかに時間がかかります。

CPU (host) and GPU (device) have physically separate memory. Python variables and NumPy arrays live in CPU memory; PyTorch tensors can live on `gs.device` (GPU). Copying between them requires a PCIe bus transfer — far slower than the computation itself.

| CPU memory (host) | PCIe bus | GPU memory (device) |
| --- | --- | --- |
| Python vars, NumPy arrays | ──► ◄── (~5-10 μs/transfer) | Tensors on `gs.device` |

**そのため、バッチテンソルはGPU上に置いたままにします。**

単一環境では、毎ステップ `qpos` をPython側に戻す必要があります（`.cpu()` によるCPUとGPUの往復 = PCIe転送）。N環境をバッチ化すると、`(N, dof)` テンソルは `gs.device` 上に留まったまま、状態の読み取りもGPU内で完結します。CPUへの転送は、結果を表示するときなど、処理の境界でのみ行います。

Single-env pulls `qpos` back to Python every step (`.cpu()` round trip = PCIe transfer). With N envs, the `(N, dof)` tensor stays on `gs.device` — state reads complete entirely on GPU. Host transfers happen only at feature boundaries (e.g. printing results).

- **繰り返し実行されるループの中で `.cpu()` や `.item()` を呼ばない / Never call `.cpu()` / `.item()` inside hot loops**

## 3. Why batched wins / なぜバッチが勝つか

| | Single-env (Python loop, N=8) | Batched (one call, N=8) |
| --- | --- | --- |
| `scene.step()` calls | 8 | 1 |
| Kernel launches | 8 | 1 |
| CPU↔GPU sync points | 8 | 1 |
| Code | `for i in range(8): scene.step()` | `scene.step()` (n_envs=8) |

- **カーネル起動のオーバーヘッドを1回にまとめる / One kernel-launch overhead:** `scene.step()` を呼ぶたびにCPUとGPUの同期が発生します。バッチ化すれば、1回の同期でN個の環境をまとめて処理できます。
- **テンソルはGPU上に留まる / Tensors stay on GPU:** `(N, dof)` テンソルは `gs.device` 上に存在したままです。繰り返し処理の中で `.cpu()` を呼ぶ必要はありません。
- **GPUの並列性を活用 / GPU parallelism:** バッチ化されたソルバはN個の環境を1つの並列処理でまとめて計算し、GPUの計算能力を十分に活かせます。

## 4. Benchmark — single vs batched / ベンチマーク

同じロボットを1環境だけ構築した場合とN環境構築した場合で、ステップあたりの実行時間とスループット（env-steps/s）を比較します。

Compare per-step time and throughput (env-steps/s) between a single-env scene and the N-env scene.

In [ ]:
import time

K = 100  # steps per benchmark


def benchmark(scene, K=100):
    """Return (ms_per_step, env_steps_per_sec) for a given built scene."""
    # Warm up (exclude first-call compilation overhead)
    for _ in range(5):
        scene.step()
    t0 = time.perf_counter()
    for _ in range(K):
        scene.step()
    elapsed = time.perf_counter() - t0
    n_envs = scene.sim.n_envs
    ms_per_step = elapsed / K * 1000
    env_steps_per_sec = n_envs * K / elapsed
    return ms_per_step, env_steps_per_sec


# --- Single-env scene -------------------------------------------------
scene1 = gs.Scene(
    sim_options=gs.options.SimOptions(dt=0.02),
    rigid_options=gs.options.RigidOptions(use_gjk_collision=True),
    show_viewer=False,
)
_ = scene1.add_entity(gs.morphs.Plane())
hsr1 = scene1.add_entity(
    HSRBURDF(file=str(tutorial_utils._find_urdf()), robot="hsrb",
             base_mode="planar",
             links_to_keep=["hand_palm_link"],
             end_effector_frame="hand_palm_link")
)
scene1.build(n_envs=1)

ms1, eps1 = benchmark(scene1, K)
print(f"Single-env:  {ms1:.2f} ms/step   {eps1:,.0f} env-steps/s")

# --- N-env scene (already built above) --------------------------------
msN, epsN = benchmark(scene, K)
print(f"{N}-env:       {msN:.2f} ms/step   {epsN:,.0f} env-steps/s")

# --- Comparison -------------------------------------------------------
speedup = epsN / eps1
print(f"\nThroughput speedup: {speedup:.1f}x  ({epsN / eps1:.1f}x more env-steps/s)")
print(f"Per-env cost:  single={ms1:.2f} ms  vs  batched={msN / N:.2f} ms/env")

## 5. What's next / 次のステップ

- **Notebook 9 (CMA-ES):** ここで学んだ並列化の仕組みを使って、把持パラメータを最適化する進化戦略を構築します。
- **Notebook 10 (PPO):** 同じ `scene.build(n_envs=N)` と `envs_idx` の仕組みを使って、観測・行動・報酬のテンソルを扱う強化学習ループを構築します。

**まとめ / Recap:**
- `scene.build(n_envs=N)` を呼ぶと、Genesisがシーンの構成をN個分複製します。これが単一環境との唯一の違いです。
- `envs_idx` は `torch.arange(N, device=gs.device)` として使います。発想を切り替えるポイントはここだけです。
- すべてのテンソルは `(N, ...)` の形状のまま `gs.device` 上に留まります。